<a href="https://colab.research.google.com/github/mbachant/Character-Complexity/blob/main/Glyph_Complexity_Calc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Glyph Complexity Database
#
# Calculates visual complexity metrics for Unicode characters
# rendered in selected fonts.
#
# The database is incremental:
#   - existing font × character measurements are retained
#   - new fonts can be added later
#   - only new font × character combinations are calculated
#
# Inputs:
#   unicode_character_database.pkl
#   noto_font_table.pkl
#   glyph_complexity_database.pkl   (if it already exists)
#
# Output:
#   glyph_complexity_database.pkl


import os
import io

import numpy as np
import pandas as pd
import cv2

from PIL import Image, ImageDraw, ImageFont
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.ndimage import distance_transform_edt
from skimage.measure import label

import pickle

from google.colab import drive, files

In [12]:
# Install Noto fonts for each Colab runtime
!apt-get update -qq
!apt-get install -y fonts-noto -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-noto-core.
(Reading database ... 118337 files and directories currently installed.)
Preparing to unpack .../0-fonts-noto-core_20201225-1build1_all.deb ...
Unpacking fonts-noto-core (20201225-1build1) ...
Selecting previously unselected package fonts-noto.
Preparing to unpack .../1-fonts-noto_20201225-1build1_all.deb ...
Unpacking fonts-noto (20201225-1build1) ...
Selecting previously unselected package fonts-noto-cjk.
Preparing to unpack .../2-fonts-noto-cjk_1%3a20220127+repack1-1_all.deb ...
Unpacking fonts-noto-cjk (1:20220127+repack1-1) ...
Selecting previously unselected package fonts-noto-cjk-extra.
Preparing to unpack .../3-fonts-noto-cjk-extra_1%3a20220127+repack1-1_all.deb ...
Unpacking fonts-noto-cjk-extra (1:20220127+repack1-1) ...
Selecting pre

In [3]:
# Choose input/output locations
#
# "drive"    = Google Drive
# "computer" = upload/download using your computer

input_source = "drive"
output_source = "drive"

base_dir = "/content/drive/MyDrive/Character Complexity"

unicode_database_filename = (
    "unicode_character_database.pkl"
)

font_table_filename = (
    "noto_font_table.pkl"
)

complexity_database_filename = (
    "glyph_complexity_database.pkl"
)

unicode_font_map_filename = (
    "unicode_to_notofonts_map.pkl"
)

In [4]:
# Paths

if input_source == "drive" or output_source == "drive":

    drive.mount(
        "/content/drive",
        force_remount=False
    )

input_dir = (
    base_dir
    if input_source == "drive"
    else "/content"
)

output_dir = (
    base_dir
    if output_source == "drive"
    else "/content"
)

unicode_database_path = os.path.join(
    input_dir,
    unicode_database_filename
)

font_table_path = os.path.join(
    input_dir,
    font_table_filename
)

complexity_database_path = os.path.join(
    output_dir,
    complexity_database_filename
)

unicode_font_map_path = os.path.join(
    input_dir,
    unicode_font_map_filename
)

print("Input directory:")
print(input_dir)

print()
print("Output directory:")
print(output_dir)

print()
print("Complexity database:")
print(complexity_database_path)

Mounted at /content/drive
Input directory:
/content/drive/MyDrive/Character Complexity

Output directory:
/content/drive/MyDrive/Character Complexity

Complexity database:
/content/drive/MyDrive/Character Complexity/glyph_complexity_database.pkl


In [7]:
# Load Unicode → Noto font map

if not os.path.isfile(unicode_font_map_path):

    raise FileNotFoundError(
        f"Unicode → font map not found:\n"
        f"{unicode_font_map_path}"
    )

with open(
    unicode_font_map_path,
    "rb"
) as f:

    unicode_to_notofonts = pickle.load(f)

print(
    f"Unicode codepoints in font map: "
    f"{len(unicode_to_notofonts):,}"
)

Unicode codepoints in font map: 79,029


In [8]:
# Load Unicode database and font table

if not os.path.isfile(unicode_database_path):
    raise FileNotFoundError(
        f"Unicode database not found:\n"
        f"{unicode_database_path}"
    )

if not os.path.isfile(font_table_path):
    raise FileNotFoundError(
        f"Font table not found:\n"
        f"{font_table_path}"
    )


character_database = pd.read_pickle(
    unicode_database_path
)

font_table = pd.read_pickle(
    font_table_path
)


# Load existing complexity database if available

if os.path.isfile(complexity_database_path):

    complexity_database = pd.read_pickle(
        complexity_database_path
    )

    database_exists = True

    print(
        f"Loaded existing database: "
        f"{len(complexity_database):,} records"
    )

else:

    complexity_database = pd.DataFrame()

    database_exists = False

    print(
        "No existing complexity database found."
    )

    print(
        "A new database will be created."
    )


# Ensure the complexity column always exists.
#
# It will be populated after PCA is calculated.

if "complexity" not in complexity_database.columns:

    complexity_database["complexity"] = np.nan

Loaded existing database: 46,000 records


In [9]:
# Verify the Unicode character database
#
# Confirm that the uploaded database contains only the
# character categories used for this analysis:
#   L = letters
#   N = numbers
#   P = punctuation
#   S = symbols
#
# Plus the selected whitespace characters:
#   U+0020 SPACE
#   U+0009 TAB
#   U+000A NEWLINE
#
# Combining marks (M) are excluded.


allowed_categories = {
    "L",
    "N",
    "P",
    "S"
}

allowed_whitespace = {
    0x0020,  # SPACE
    0x0009,  # TAB
    0x000A   # NEWLINE
}


# Check for unexpected characters

unexpected = character_database[
    ~(
        character_database["category"].str[0].isin(
            allowed_categories
        )
        |
        character_database["codepoint"].isin(
            allowed_whitespace
        )
    )
]


if not unexpected.empty:

    raise ValueError(
        f"Database contains {len(unexpected):,} "
        "characters outside the specified parameters."
    )


# Confirm required categories are present

categories_present = set(
    character_database["category"].str[0]
)

missing_categories = (
    allowed_categories
    - categories_present
)


if missing_categories:

    raise ValueError(
        f"Database is missing categories: "
        f"{sorted(missing_categories)}"
    )


print(
    "Unicode database verified."
)

print(
    f"Characters: {len(character_database):,}"
)

print(
    f"Categories: "
    f"{sorted(categories_present)}"
)

print(
    "Additional whitespace: SPACE, TAB, NEWLINE"
)

Unicode database verified.
Characters: 146,550
Categories: ['C', 'L', 'N', 'P', 'S', 'Z']
Additional whitespace: SPACE, TAB, NEWLINE


In [10]:
# Font lookup
# Edit this to search for a font name if one is preferred

font_search = "Noto Sans"

font_lookup = font_table[
    font_table["font_name"]
    .str.contains(
        font_search,
        case=False,
        na=False
    )
].copy()

display(
    font_lookup[
        [
            "font_id",
            "font_name",
            "family_name",
            "font_path",
            "font_fileindex"
        ]
    ]
)


,font_id,font_name,family_name,font_path,font_fileindex
0,0,Noto Sans Mono Condensed Bold,Noto Sans Mono Condensed,/usr/share/fonts/truetype/noto/NotoSansMono-Co...,0
1,1,Noto Sans Georgian Condensed Medium,Noto Sans Georgian Cond Med,/usr/share/fonts/truetype/noto/NotoSansGeorgia...,0
2,2,Noto Sans Light Italic,Noto Sans Light,/usr/share/fonts/truetype/noto/NotoSans-LightI...,0
3,3,Noto Sans Canadian Aboriginal Medium,Noto Sans CanAborig Md,/usr/share/fonts/truetype/noto/NotoSansCanadia...,0
5,5,Noto Sans CJK JP Medium,Noto Sans CJK JP Medium,/usr/share/fonts/opentype/noto/NotoSansCJK-Med...,0
...,...,...,...,...,...
2454,2454,Noto Sans Arabic ExtraCondensed ExtraBold,Noto Sans Arabic ExtCond ExtBd,/usr/share/fonts/truetype/noto/NotoSansArabic-...,0
2455,2455,Noto Sans Malayalam UI SemiCondensed,Noto Sans Malayalam UI SemiCondensed,/usr/share/fonts/truetype/noto/NotoSansMalayal...,0
2457,2457,Noto Sans Kannada Condensed,Noto Sans Kannada Condensed,/usr/share/fonts/truetype/noto/NotoSansKannada...,0
2458,2458,Noto Sans Lao UI SemiCondensed,Noto Sans Lao UI SemCond,/usr/share/fonts/truetype/noto/NotoSansLaoUI-S...,0


In [13]:
# Select fonts to add to the complexity database

Rendering_fonts = [
    1344,
    2211,
    2213,
    2288

]

selected_fonts = font_table[
    font_table["font_id"].isin(Rendering_fonts)
].copy()

missing_fonts = set(Rendering_fonts) - set(
    selected_fonts["font_id"]
)

if missing_fonts:
    raise ValueError(
        f"Font IDs not found in font_table: "
        f"{sorted(missing_fonts)}"
    )

for _, font in selected_fonts.iterrows():

    if not os.path.isfile(font["font_path"]):
        raise FileNotFoundError(
            f"Font file not found:\n"
            f"{font['font_path']}"
        )

print("Fonts selected:")

display(
    selected_fonts[
        [
            "font_id",
            "font_name",
            "font_path",
            "font_fileindex"
        ]
    ]
)

Fonts selected:


,font_id,font_name,font_path,font_fileindex
1344,1344,Noto Sans Mono Condensed Light,/usr/share/fonts/truetype/noto/NotoSansMono-Co...,0
2211,2211,Noto Sans Mono ExtraCondensed Thin,/usr/share/fonts/truetype/noto/NotoSansMono-Ex...,0
2213,2213,Noto Sans Mono ExtraCondensed SemiBold,/usr/share/fonts/truetype/noto/NotoSansMono-Ex...,0
2288,2288,Noto Sans Mono Condensed Black,/usr/share/fonts/truetype/noto/NotoSansMono-Co...,0


In [14]:
# Find characters supported by the selected fonts

selected_font_ids = set(
    selected_fonts["font_id"]
)

renderable_characters = (
    character_database[
        ["character", "codepoint", "unicode", "unicode_name", "category"]
    ]
    .copy()
)

renderable_characters["font_ids"] = (
    renderable_characters["codepoint"]
    .map(
        lambda codepoint:
            unicode_to_notofonts.get(
                codepoint,
                {}
            ).get(
                "font_ids",
                []
            )
    )
)

renderable_characters = (
    renderable_characters
    .explode("font_ids")
    .rename(
        columns={
            "font_ids": "font_id"
        }
    )
)

renderable_characters = (
    renderable_characters[
        renderable_characters["font_id"].isin(
            selected_font_ids
        )
    ]
    .copy()
)

renderable_characters = renderable_characters.merge(
    selected_fonts[
        ["font_id", "font_name"]
    ],
    on="font_id",
    how="left",
    validate="many_to_one"
)

renderable_characters = renderable_characters[
    [
        "font_id",
        "font_name",
        "character",
        "codepoint",
        "unicode",
        "unicode_name",
        "category"
    ]
].reset_index(drop=True)


# Summary

total_characters = len(character_database)

print(
    f"Characters in Unicode database: "
    f"{total_characters:,}"
)

print(
    f"Selected fonts: "
    f"{len(selected_fonts):,}"
)

print(
    f"Renderable font × character combinations: "
    f"{len(renderable_characters):,}"
)


print()
print("Coverage by font:")

coverage = (
    renderable_characters
    .groupby(
        ["font_id", "font_name"]
    )
    .size()
    .reset_index(name="renderable")
)

coverage["total"] = total_characters

coverage["% coverage of Unicode DB"] = (
    100
    * coverage["renderable"]
    / coverage["total"]
)

display(coverage)

Characters in Unicode database: 146,550
Selected fonts: 4
Renderable font × character combinations: 12,228

Coverage by font:


,font_id,font_name,renderable,total,% coverage of Unicode DB
0,1344,Noto Sans Mono Condensed Light,3057,146550,2.085977
1,2211,Noto Sans Mono ExtraCondensed Thin,3057,146550,2.085977
2,2213,Noto Sans Mono ExtraCondensed SemiBold,3057,146550,2.085977
3,2288,Noto Sans Mono Condensed Black,3057,146550,2.085977


In [15]:
render_parameters = {
    "font_size": 200,
    "canvas_size": (512, 512),
    "background_value": 255,
    "foreground_value": 0,
    "threshold": 0.5,
    "jpeg_size": (64, 64)
}

In [16]:
# Load selected fonts

loaded_fonts = {}

for _, font in selected_fonts.iterrows():

    loaded_fonts[font["font_id"]] = ImageFont.truetype(
        font["font_path"],
        render_parameters["font_size"],
        index=int(font["font_fileindex"])
    )

print(
    f"Loaded fonts: {len(loaded_fonts):,}"
)

for _, font in selected_fonts.iterrows():

    print(
        f"{font['font_id']}: {font['font_name']}"
    )

Loaded fonts: 4
1344: Noto Sans Mono Condensed Light
2211: Noto Sans Mono ExtraCondensed Thin
2213: Noto Sans Mono ExtraCondensed SemiBold
2288: Noto Sans Mono Condensed Black


In [17]:
# Render one glyph
#
# The image is created temporarily for measurement.
# The image itself is not stored in the complexity database.

def render_glyph(
    character,
    font,
    parameters
):

    canvas_width, canvas_height = parameters[
        "canvas_size"
    ]

    image = Image.new(
        "L",
        (canvas_width, canvas_height),
        parameters["background_value"]
    )

    draw = ImageDraw.Draw(image)

    ascent, descent = font.getmetrics()

    baseline_y = (
        canvas_height
        + ascent
        - descent
    ) // 2

    draw.text(
        (
            canvas_width // 2,
            baseline_y
        ),
        character,
        font=font,
        fill=parameters["foreground_value"],
        anchor="ms"
    )

    grayscale_array = (
        np.asarray(image)
        .astype(np.float32)
        / 255.0
    )

    binary_array = (
        grayscale_array
        < parameters["threshold"]
    )

    if np.any(binary_array):

        rows, cols = np.where(
            binary_array
        )

        ink_bbox = (
            int(cols.min()),
            int(rows.min()),
            int(cols.max() + 1),
            int(rows.max() + 1)
        )

    else:

        ink_bbox = None

    advance_width = draw.textlength(
        character,
        font=font
    )

    return {
        "grayscale_array": grayscale_array,
        "binary_array": binary_array,
        "ascent": ascent,
        "descent": descent,
        "baseline_y": baseline_y,
        "advance_width": advance_width,
        "ink_bbox": ink_bbox
    }

In [18]:
# Define Complexity metric functions

def crop_to_ink(binary, grayscale):

    if not np.any(binary):
        return binary, grayscale

    rows, cols = np.where(binary)

    row_min = rows.min()
    row_max = rows.max() + 1
    col_min = cols.min()
    col_max = cols.max() + 1

    return (
        binary[row_min:row_max, col_min:col_max],
        grayscale[row_min:row_max, col_min:col_max]
    )


def calculate_pixel_count(binary):
    return int(np.sum(binary))


def calculate_perimeters(binary):

    image = (
        binary.astype(np.uint8) * 255
    )

    contours, _ = cv2.findContours(
        image,
        cv2.RETR_CCOMP,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return 0.0, 0

    perimeter = sum(
        cv2.arcLength(contour, True)
        for contour in contours
    )

    return float(perimeter), len(contours)


def calculate_perimetric_complexity(
    area,
    perimeter
):

    if area == 0:
        return 0.0

    return (
        perimeter ** 2
        / (4 * np.pi * area)
    )


def calculate_compression_size(grayscale):

    image = (
        grayscale * 255
    ).astype(np.uint8)

    image = Image.fromarray(
        image
    ).resize(
        render_parameters["jpeg_size"],
        Image.Resampling.LANCZOS
    )

    buffer = io.BytesIO()

    image.save(
        buffer,
        format="JPEG"
    )

    return len(buffer.getvalue())


def calculate_symmetry(
    grayscale,
    axis
):

    if axis == "vertical":

        # Reflect left ↔ right across the vertical
        # central axis.
        flipped = np.fliplr(
            grayscale
        )

    elif axis == "horizontal":

        # Reflect top ↔ bottom across the horizontal
        # central axis.
        flipped = np.flipud(
            grayscale
        )

    else:

        raise ValueError(
            "axis must be 'vertical' or 'horizontal'"
        )

    # Compare the original and reflected glyph only
    # where either image contains ink.
    mask = (
        (grayscale < 0.5)
        |
        (flipped < 0.5)
    )

    if not np.any(mask):
        return 1.0

    difference = np.abs(
        grayscale - flipped
    )

    return float(
        1 - np.mean(
            difference[mask]
        )
    )


def calculate_connected_components(binary):

    labeled = label(
        binary,
        connectivity=2
    )

    return int(
        labeled.max()
    )


def calculate_distance_transform_mean(binary):

    if not np.any(binary):
        return 0.0

    distance = distance_transform_edt(
        binary
    )

    return float(
        np.mean(
            distance[binary]
        )
    )


metrics = [
    "pixel_count",
    "perimeter_len",
    "num_perimeters",
    "perimetric_complexity",
    "compression_size",
    "vertical_axis_symmetry",
    "horizontal_axis_symmetry",
    "num_components",
    "distance_transform_mean"
]

In [19]:
# Determine which font × character combinations still need to be rendered

if complexity_database.empty:

    # No existing database.
    # Therefore every renderable font × character combination
    # needs to be calculated.

    render_queue = (
        renderable_characters
        .copy()
        .reset_index(drop=True)
    )

else:

    # Existing database.
    # Only calculate font × character combinations that
    # are not already present.

    existing_keys_df = (
        complexity_database[
            ["font_id", "codepoint"]
        ]
        .drop_duplicates()
    )

    render_queue = (
        renderable_characters
        .merge(
            existing_keys_df,
            on=["font_id", "codepoint"],
            how="left",
            indicator=True
        )
    )

    render_queue = (
        render_queue[
            render_queue["_merge"] == "left_only"
        ]
        .drop(columns="_merge")
        .reset_index(drop=True)
    )


total_supported = len(
    renderable_characters
)

already_rendered = (
    total_supported
    - len(render_queue)
)

print(
    f"Supported font × character combinations: "
    f"{total_supported:,}"
)

print(
    f"Already calculated: "
    f"{already_rendered:,}"
)

print(
    f"To calculate: "
    f"{len(render_queue):,}"
)

Supported font × character combinations: 12,228
Already calculated: 0
To calculate: 12,228


In [20]:
# Calculate all metrics for one glyph

def calculate_glyph_metrics(
    character,
    font,
    parameters
):

    # Render the glyph.
    rendered = render_glyph(
        character,
        font,
        parameters
    )

    grayscale = rendered[
        "grayscale_array"
    ]

    binary = rendered[
        "binary_array"
    ]

    # Crop to the actual rendered ink.
    # Intrinsic shape metrics are therefore independent
    # of the glyph's position on the full canvas.
    intrinsic_binary, intrinsic_grayscale = crop_to_ink(
        binary,
        grayscale
    )

    # Pixel count.
    pixel_count = calculate_pixel_count(
        intrinsic_binary
    )

    # Total contour length and number of contours.
    perimeter_len, num_perimeters = calculate_perimeters(
        intrinsic_binary
    )

    # Perimeter relative to glyph area.
    perimetric_complexity = calculate_perimetric_complexity(
        pixel_count,
        perimeter_len
    )

    return {

        # Complexity metrics

        "pixel_count":
            pixel_count,

        "perimeter_len":
            perimeter_len,

        "num_perimeters":
            num_perimeters,

        "perimetric_complexity":
            perimetric_complexity,

        "compression_size":
            calculate_compression_size(
                intrinsic_grayscale
            ),

        "vertical_axis_symmetry":
            calculate_symmetry(
                intrinsic_grayscale,
                "vertical"
            ),

        "horizontal_axis_symmetry":
            calculate_symmetry(
                intrinsic_grayscale,
                "horizontal"
            ),

        "num_components":
            calculate_connected_components(
                intrinsic_binary
            ),

        "distance_transform_mean":
            calculate_distance_transform_mean(
                intrinsic_binary
            ),

        # Rendering / typography measurements

        "ascent":
            rendered["ascent"],

        "descent":
            rendered["descent"],

        "baseline_y":
            rendered["baseline_y"],

        "advance_width":
            rendered["advance_width"],

        "ink_bbox":
            rendered["ink_bbox"]
    }

In [21]:
# Create one complete database record for a glyph.
#
# Takes one row from render_queue and its loaded font,
# calculates the glyph's measurements, and combines those
# measurements with the font and Unicode metadata.

def calculate_record(row, font):

    glyph_metrics = calculate_glyph_metrics(
        row["character"],
        font,
        render_parameters
    )

    return {
        "font_id": row["font_id"],
        "font_name": row["font_name"],
        "codepoint": row["codepoint"],
        "character": row["character"],
        "unicode": row["unicode"],
        "unicode_name": row["unicode_name"],
        "category": row["category"],
        **glyph_metrics
    }

In [22]:
# Calculate missing glyphs and save the database every 1,000 records.
#
# The database is saved after every completed batch.
# A temporary file is written first and then replaces the
# previous database, reducing the risk of leaving a corrupted
# pickle if Colab is interrupted during saving.

batch_size = 1000


def save_complexity_database(database):

    temporary_path = (
        complexity_database_path
        + ".tmp"
    )

    database.to_pickle(
        temporary_path
    )

    os.replace(
        temporary_path,
        complexity_database_path
    )


for start in range(
    0,
    len(render_queue),
    batch_size
):

    batch = render_queue.iloc[
        start:start + batch_size
    ]

    batch_records = []

    for _, row in batch.iterrows():

        font = loaded_fonts[
            row["font_id"]
        ]

        batch_records.append(
            calculate_record(
                row,
                font
            )
        )

    batch_df = pd.DataFrame(
        batch_records
    )

    complexity_database = pd.concat(
        [
            complexity_database,
            batch_df
        ],
        ignore_index=True
    )

    complexity_database = (
        complexity_database
        .drop_duplicates(
            subset=[
                "font_id",
                "codepoint"
            ],
            keep="first"
        )
        .reset_index(drop=True)
    )

    # Make absolutely sure the column exists.

    if "complexity" not in complexity_database.columns:

        complexity_database["complexity"] = np.nan

    # Safe save.

    save_complexity_database(
        complexity_database
    )

    processed = min(
        start + batch_size,
        len(render_queue)
    )

    print(
        f"Saved: {processed:,} / "
        f"{len(render_queue):,}"
    )


print()
print("Glyph rendering complete.")

print(
    f"Total glyph × font records: "
    f"{len(complexity_database):,}"
)

print(
    f"Fonts in database: "
    f"{complexity_database['font_id'].nunique():,}"
)

print(
    f"Unique characters: "
    f"{complexity_database['codepoint'].nunique():,}"
)

print(
    f"Complexity column present: "
    f"{'complexity' in complexity_database.columns}"
)

Saved: 1,000 / 12,228
Saved: 2,000 / 12,228
Saved: 3,000 / 12,228
Saved: 4,000 / 12,228
Saved: 5,000 / 12,228
Saved: 6,000 / 12,228
Saved: 7,000 / 12,228
Saved: 8,000 / 12,228
Saved: 9,000 / 12,228
Saved: 10,000 / 12,228
Saved: 11,000 / 12,228
Saved: 12,000 / 12,228
Saved: 12,228 / 12,228

Glyph rendering complete.
Total glyph × font records: 58,228
Fonts in database: 8
Unique characters: 48,242
Complexity column present: True


In [ ]:
# # IF NECESSARY:
# ## Recalculate EVERY glyph in the complexity database
# #
# # This deliberately ignores existing metric values and
# # re-renders every font × character combination.

# recalculation_queue = (
#     complexity_database[
#         [
#             "font_id",
#             "font_name",
#             "codepoint",
#             "character",
#             "unicode",
#             "unicode_name",
#             "category"
#         ]
#     ]
#     .drop_duplicates(
#         subset=["font_id", "codepoint"]
#     )
#     .copy()
# )

# print(
#     f"Glyphs to recalculate: "
#     f"{len(recalculation_queue):,}"
# )


# # Load the fonts required for recalculation

# loaded_fonts = {}

# fonts_needed = (
#     recalculation_queue[
#         ["font_id", "font_name"]
#     ]
#     .drop_duplicates()
#     .merge(
#         selected_fonts[
#             [
#                 "font_id",
#                 "font_path",
#                 "font_fileindex"
#             ]
#         ],
#         on="font_id",
#         how="left"
#     )
# )

# for _, row in fonts_needed.iterrows():

#     font_id = row["font_id"]

#     if not os.path.isfile(row["font_path"]):

#         raise FileNotFoundError(
#             f"Font file not found:\n"
#             f"{row['font_path']}"
#         )

#     loaded_fonts[font_id] = ImageFont.truetype(
#         row["font_path"],
#         render_parameters["font_size"],
#         index=int(row["font_fileindex"])
#     )


# # Recalculate metrics

# recalculated_records = []

# for _, row in recalculation_queue.iterrows():

#     font = loaded_fonts[row["font_id"]]

#     rendered = render_glyph(
#         row["character"],
#         font,
#         render_parameters
#     )

#     grayscale_array = rendered["grayscale_array"]
#     binary_array = rendered["binary_array"]

#     intrinsic_binary, intrinsic_grayscale = crop_to_ink(
#         binary_array,
#         grayscale_array
#     )

#     area = calculate_pixel_count(
#         intrinsic_binary
#     )

#     perimeter_len, num_perimeters = calculate_perimeters(
#         intrinsic_binary
#     )

#     perimetric_complexity = (
#         perimeter_len ** 2
#         / (4 * np.pi * area)
#         if area > 0
#         else 0.0
#     )

#     compression_size = calculate_compression_size(
#         intrinsic_grayscale
#     )

#     horizontal_symmetry = calculate_symmetry(
#         intrinsic_grayscale,
#         "horizontal"
#     )

#     vertical_symmetry = calculate_symmetry(
#         intrinsic_grayscale,
#         "vertical"
#     )

#     num_components = calculate_connected_components(
#         intrinsic_binary
#     )

#     distance_transform_mean = (
#         calculate_distance_transform_mean(
#             intrinsic_binary
#         )
#     )

#     recalculated_records.append({

#         "font_id": row["font_id"],
#         "font_name": row["font_name"],
#         "codepoint": row["codepoint"],
#         "character": row["character"],
#         "unicode": row["unicode"],
#         "unicode_name": row["unicode_name"],
#         "category": row["category"],

#         "pixel_count": area,
#         "perimeter_len": perimeter_len,
#         "num_perimeters": num_perimeters,
#         "perimetric_complexity": perimetric_complexity,
#         "compression_size": compression_size,
#         "horizontal_symmetry": horizontal_symmetry,
#         "vertical_symmetry": vertical_symmetry,
#         "num_components": num_components,
#         "distance_transform_mean": distance_transform_mean,

#         "ascent": rendered["ascent"],
#         "descent": rendered["descent"],
#         "baseline_y": rendered["baseline_y"],
#         "advance_width": rendered["advance_width"],
#         "ink_bbox": rendered["ink_bbox"]
#     })


# recalculated_db = pd.DataFrame(
#     recalculated_records
# )

# print(
#     f"Recalculated: "
#     f"{len(recalculated_db):,}"
# )

# complexity_database = recalculated_db

In [24]:
# Calculate global PCA-derived glyph complexity score
#
# PCA is fitted across the complete glyph complexity database.
# The resulting normalized PC1 score is stored as "complexity".

metrics = [
    "pixel_count",
    "perimeter_len",
    "num_perimeters",
    "perimetric_complexity",
    "compression_size",
    "vertical_axis_symmetry",
    "horizontal_axis_symmetry",
    "num_components",
    "distance_transform_mean"
]


# Always initialize the column.

complexity_database["complexity"] = np.nan


metric_data = (
    complexity_database[
        metrics
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


valid_mask = (
    metric_data
    .notna()
    .all(axis=1)
)


metric_data = metric_data[
    valid_mask
]


if len(metric_data) < 2:

    raise ValueError(
        "Not enough complete glyph measurements "
        "to calculate PCA complexity."
    )


# Standardize metrics.

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    metric_data
)


# PCA.

pca = PCA(
    n_components=1
)

scores = pca.fit_transform(
    X_scaled
)[:, 0]


# Orient PC1 so that greater perimeter generally
# corresponds to greater complexity.

if pd.Series(scores).corr(
    metric_data["perimeter_len"]
) < 0:

    scores *= -1


# Normalize complexity to 0–1.

minimum = scores.min()
maximum = scores.max()


if maximum > minimum:

    scores = (
        scores - minimum
    ) / (
        maximum - minimum
    )

else:

    scores = np.zeros(
        len(scores)
    )


# Store scores back in the full database.

complexity_database.loc[
    metric_data.index,
    "complexity"
] = scores


print()
print(
    f"PC1 explained variance: "
    f"{pca.explained_variance_ratio_[0]:.1%}"
)

print(
    f"Glyphs with complexity scores: "
    f"{complexity_database['complexity'].notna().sum():,}"
)

print(
    f"Glyphs without complexity scores: "
    f"{complexity_database['complexity'].isna().sum():,}"
)


# PCA loadings

loadings = pd.Series(
    pca.components_[0],
    index=metrics
).sort_values(
    ascending=False
)

display(loadings)


PC1 explained variance: 54.2%
Glyphs with complexity scores: 58,228
Glyphs without complexity scores: 0


,0
perimeter_len,0.432848
compression_size,0.412801
perimetric_complexity,0.405832
num_perimeters,0.384774
pixel_count,0.379982
num_components,0.341108
vertical_axis_symmetry,-0.138852
distance_transform_mean,-0.148537
horizontal_axis_symmetry,-0.165539


In [25]:
# Inspect the dimensions and basic contents of the updated
# glyph-complexity database.

print(
    f"Database dimensions: "
    f"{complexity_database.shape[0]:,} rows × "
    f"{complexity_database.shape[1]:,} columns"
)

print()
print("Columns:")
print(
    list(complexity_database.columns)
)

print()
print("Unique fonts:")
print(
    complexity_database["font_id"].nunique()
)

print()
print("Unique characters:")
print(
    complexity_database["codepoint"].nunique()
)

print()
print("Non-null complexity values:")
print(
    complexity_database["complexity"].notna().sum()
)

print()
print("Missing complexity values:")
print(
    complexity_database["complexity"].isna().sum()
)
print()
print("Records per font:")

display(
    complexity_database
    .groupby(
        ["font_id", "font_name"]
    )
    .size()
    .reset_index(name="records")
    .sort_values(
        "records",
        ascending=False
    )
    .reset_index(drop=True)
)
display(
    complexity_database.head(10)
)

Database dimensions: 58,228 rows × 22 columns

Columns:
['font_id', 'font_name', 'codepoint', 'character', 'unicode', 'unicode_name', 'category', 'pixel_count', 'perimeter_len', 'num_perimeters', 'perimetric_complexity', 'compression_size', 'vertical_axis_symmetry', 'horizontal_axis_symmetry', 'num_components', 'distance_transform_mean', 'ascent', 'descent', 'baseline_y', 'advance_width', 'ink_bbox', 'complexity']

Unique fonts:
8

Unique characters:
48242

Non-null complexity values:
58228

Missing complexity values:
0

Records per font:


,font_id,font_name,records
0,2142,Noto Sans CJK SC,44790
1,1344,Noto Sans Mono Condensed Light,3057
2,2211,Noto Sans Mono ExtraCondensed Thin,3057
3,2213,Noto Sans Mono ExtraCondensed SemiBold,3057
4,2288,Noto Sans Mono Condensed Black,3057
5,1667,Noto Sans Arabic Regular,1047
6,1050,Noto Sans Hebrew Regular,86
7,827,Noto Sans Thai Regular,77


,font_id,font_name,codepoint,character,unicode,unicode_name,category,pixel_count,perimeter_len,num_perimeters,...,vertical_axis_symmetry,horizontal_axis_symmetry,num_components,distance_transform_mean,ascent,descent,baseline_y,advance_width,ink_bbox,complexity
0,1667,Noto Sans Arabic Regular,33,!,U+0021,EXCLAMATION MARK,Po,2237,323.740114,2,...,0.931747,0.658305,2,4.675662,275,148,319,53.796875,"(243, 176, 268, 322)",0.166244
1,2142,Noto Sans CJK SC,33,!,U+0021,EXCLAMATION MARK,Po,2227,326.083260,2,...,0.962670,0.629747,2,4.552383,232,58,343,64.593750,"(243, 194, 269, 346)",0.163232
2,2142,Noto Sans CJK SC,34,"""",U+0022,QUOTATION MARK,Po,1890,291.455844,2,...,0.953740,0.757625,2,5.243369,232,58,343,94.796875,"(227, 189, 286, 247)",0.157165
3,2142,Noto Sans CJK SC,35,#,U+0023,NUMBER SIGN,Po,5227,862.894442,2,...,0.431655,0.337165,1,3.709714,232,58,343,111.000000,"(208, 198, 304, 343)",0.222214
4,2142,Noto Sans CJK SC,36,$,U+0024,DOLLAR SIGN,Sc,5148,705.997035,1,...,0.396309,0.411891,1,4.358484,232,58,343,111.000000,"(212, 176, 298, 367)",0.207263
5,2142,Noto Sans CJK SC,37,%,U+0025,PERCENT SIGN,Po,7407,1301.869182,5,...,0.246116,0.245929,3,3.604102,232,58,343,184.203125,"(172, 194, 340, 346)",0.279606
6,2142,Noto Sans CJK SC,38,&,U+0026,AMPERSAND,Po,7286,964.714853,3,...,0.194568,0.333802,1,4.358837,232,58,343,136.000000,"(195, 194, 320, 346)",0.247304
7,2142,Noto Sans CJK SC,39,',U+0027,APOSTROPHE,Po,947,145.313708,1,...,0.980507,0.761497,1,8.335744,232,58,343,55.593750,"(246, 189, 265, 247)",0.127166
8,2142,Noto Sans CJK SC,40,(,U+0028,LEFT PARENTHESIS,Ps,2922,462.877198,1,...,0.124643,0.981707,1,3.886479,232,58,343,67.593750,"(240, 180, 281, 382)",0.175803
9,2142,Noto Sans CJK SC,41,),U+0029,RIGHT PARENTHESIS,Pe,2911,463.462984,1,...,0.121877,0.990329,1,4.166812,232,58,343,67.593750,"(231, 179, 271, 382)",0.175397


In [27]:
# Final safe save

temporary_path = (
    complexity_database_path
    + ".tmp"
)

complexity_database.to_pickle(
    temporary_path
)

os.replace(
    temporary_path,
    complexity_database_path
)


# Verify the saved file can actually be reopened.

verified_database = pd.read_pickle(
    complexity_database_path
)


required_columns = {
    "font_id",
    "codepoint",
    "character",
    "complexity"
}

missing_columns = (
    required_columns
    - set(verified_database.columns)
)


if missing_columns:

    raise ValueError(
        "Saved database is missing required columns: "
        f"{sorted(missing_columns)}"
    )


print()
print("=" * 60)
print("DATABASE SAVED AND VERIFIED")
print("=" * 60)

print()
print(
    f"Path: {complexity_database_path}"
)

print(
    f"Records: {len(verified_database):,}"
)

print(
    f"Fonts: "
    f"{verified_database['font_id'].nunique():,}"
)

print(
    f"Characters: "
    f"{verified_database['codepoint'].nunique():,}"
)

print(
    f"Complexity values: "
    f"{verified_database['complexity'].notna().sum():,}"
)

print()
print("Required columns present:")
print(
    sorted(required_columns)
)


DATABASE SAVED AND VERIFIED

Path: /content/drive/MyDrive/Character Complexity/glyph_complexity_database.pkl
Records: 58,228
Fonts: 8
Characters: 48,242
Complexity values: 58,228

Required columns present:
['character', 'codepoint', 'complexity', 'font_id']
